# SageMaker V3 Custom Distributed Training Example

This notebook demonstrates how to create and use custom distributed training drivers with SageMaker V3 ModelTrainer.

In [1]:
import os
import tempfile
import shutil

from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode
from sagemaker.train.distributed import DistributedConfig
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## Step 1: Setup Session and Create Test Files

Initialize the SageMaker session and create the custom distributed driver files.

In [2]:
sagemaker_session = Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name

DEFAULT_CPU_IMAGE = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.0.0",
    py_version="py310",
    instance_type="ml.m5.xlarge",
    image_scope="training"
)

# Create temporary directories
temp_dir = tempfile.mkdtemp()
custom_drivers_dir = os.path.join(temp_dir, "custom_drivers")
scripts_dir = os.path.join(temp_dir, "scripts")

os.makedirs(custom_drivers_dir, exist_ok=True)
os.makedirs(scripts_dir, exist_ok=True)

print(f"Created temporary directories in: {temp_dir}")

[07/16/26 07:11:56] WARNING  Couldn't call 'get_role' to get Role ARN from role name          ]8;id=15208139;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=15208140;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#375\375]8;;\
                             NotebookTestEngine-ProcessingJobRole to get Role path.                                

Created temporary directories in: /home/model-server/tmp/tmpoab_oo4k


## Step 2: Create Custom Driver and Entry Script

Create the custom driver script and entry script for training.

In [3]:
# Create custom driver script
driver_script = '''
import json
import os
import subprocess
import sys

def main():
    driver_config = json.loads(os.environ["SM_DISTRIBUTED_CONFIG"])
    process_count_per_node = driver_config["process_count_per_node"]
    assert process_count_per_node != None

    hps = json.loads(os.environ["SM_HPS"])
    assert hps != None
    assert isinstance(hps, dict)

    source_dir = os.environ["SM_SOURCE_DIR"]
    assert source_dir == "/opt/ml/input/data/code"
    sm_drivers_dir = os.environ["SM_DISTRIBUTED_DRIVER_DIR"]
    assert sm_drivers_dir == "/opt/ml/input/data/sm_drivers/distributed_drivers"

    entry_script = os.environ["SM_ENTRY_SCRIPT"]
    assert entry_script != None

    python = sys.executable

    command = [python, entry_script]
    print(f"Running command: {command}")
    subprocess.run(command, check=True)

if __name__ == "__main__":
    print("Running custom driver script")
    main()
    print("Finished running custom driver script")
'''

with open(os.path.join(custom_drivers_dir, "driver.py"), 'w') as f:
    f.write(driver_script)

print("Created custom driver script")

Created custom driver script


In [4]:
# Create entry script
entry_script = '''
import json
import os
import time

def main():
    hps = json.loads(os.environ["SM_HPS"])
    assert hps != None
    print(f"Hyperparameters: {hps}")

    print("Running pseudo training script")
    for epochs in range(hps["epochs"]):
        print(f"Epoch: {epochs}")
        time.sleep(1)
    print("Finished running pseudo training script")
    
    # Save results
    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    os.makedirs(model_dir, exist_ok=True)
    
    results = {"status": "success", "epochs_completed": hps["epochs"]}
    with open(os.path.join(model_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

if __name__ == "__main__":
    main()
'''

with open(os.path.join(scripts_dir, "entry_script.py"), 'w') as f:
    f.write(entry_script)

print("Created entry script")

Created entry script


## Step 3: Define Custom Distributed Driver

Create the custom distributed driver class.

In [5]:
class CustomDriver(DistributedConfig):
    process_count_per_node: int = None

    @property
    def driver_dir(self) -> str:
        return custom_drivers_dir

    @property
    def driver_script(self) -> str:
        return "driver.py"

print("Custom distributed driver class defined!")
print(f"Driver directory: {custom_drivers_dir}")
print(f"Driver script: driver.py")

Custom distributed driver class defined!
Driver directory: /home/model-server/tmp/tmpoab_oo4k/custom_drivers
Driver script: driver.py


## Step 4: Configure Source Code and Hyperparameters

Set up the source code and hyperparameters for training.

In [6]:
source_code = SourceCode(
    source_dir=scripts_dir,
    entry_script="entry_script.py",
)

hyperparameters = {"epochs": 10}

custom_driver = CustomDriver(process_count_per_node=2)

print(f"Source directory: {scripts_dir}")
print(f"Entry script: entry_script.py")
print(f"Hyperparameters: {hyperparameters}")
print(f"Custom driver: {custom_driver}")

Source directory: /home/model-server/tmp/tmpoab_oo4k/scripts
Entry script: entry_script.py
Hyperparameters: {'epochs': 10}
Custom driver: process_count_per_node=2


## Step 5: Create ModelTrainer with Custom Driver

Initialize ModelTrainer with the custom distributed configuration.

In [7]:
model_trainer = ModelTrainer(
    sagemaker_session=sagemaker_session,
    training_image=DEFAULT_CPU_IMAGE,
    hyperparameters=hyperparameters,
    source_code=source_code,
    distributed=custom_driver,
    base_job_name="custom-distributed-driver",
)

print("ModelTrainer created with custom distributed driver!")
print(f"Job name: custom-distributed-driver")
print(f"Distributed configuration: {model_trainer.distributed}")

                    INFO     Cannot simulate policies for                                  ]8;id=15208147;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=15208148;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=15208154;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=15208155;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Role not provided. Using validated caller role:                         ]8;id=15208162;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=15208163;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole                   

                    INFO     Compute not provided. Using default:                                   ]8;id=15208169;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=15208170;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#141\141]8;;\
                             instance_type='ml.m5.xlarge' instance_count=1 volume_size_in_gb=30                    
                             volume_kms_key_id=None keep_alive_period_in_seconds=None                              
                             instance_groups=None training_plan_arn=None                                           
                             instance_placement_config=None enable_managed_spot_training=None                      

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=15208176;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=15208177;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

[07/16/26 07:11:57] INFO     OutputDataConfig not provided. Using default:                          ]8;id=15208183;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=15208184;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-674622101542/custom-distribut                
                             ed-driver' kms_key_id=None compression_type='GZIP'                                    

                    INFO     Training image URI:                                               ]8;id=15208191;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=15208192;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.0                     
                             .0-cpu-py310                                                                          

ModelTrainer created with custom distributed driver!
Job name: custom-distributed-driver
Distributed configuration: process_count_per_node=2


## Step 6: Run Custom Distributed Training

Start the distributed training job using the custom driver.

In [8]:
print("Starting custom distributed training...")

try:
    model_trainer.train()
    print(f"Custom distributed training completed successfully!")
    print(f"Job name: {model_trainer._latest_training_job.training_job_name}")
    training_successful = True
except Exception as e:
    print(f"Training failed with error: {e}")
    training_successful = False

Starting custom distributed training...


                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=15208199;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=15208200;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[07/16/26 07:11:59] INFO     Creating training_job resource.                                     ]8;id=15208207;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15208208;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31239\31239]8;;\

...

                    INFO     custom-distributed-driver-20260716071158/algo-1-1784185953:         ]8;id=15208793;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15208794;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Finished running custom driver script                                                 

                    INFO     custom-distributed-driver-20260716071158/algo-1-1784185953:         ]8;id=15208799;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15208800;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     custom-distributed-driver-20260716071158/algo-1-1784185953:         ]8;id=15208805;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15208806;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Training Container Execution Completed                                                

[07/16/26 07:14:52] INFO     Final Resource Status: Completed                                    ]8;id=15208812;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15208813;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31591\31591]8;;\

Custom distributed training completed successfully!
Job name: custom-distributed-driver-20260716071158


## Step 7: Analyze Training Results

Examine the results from the custom distributed training.

In [9]:
if training_successful:
    job_name = model_trainer._latest_training_job.training_job_name
    model_artifacts = model_trainer._latest_training_job.model_artifacts
    
    print("Custom Distributed Training Results:")
    print("=" * 40)
    print(f"Job Name: {job_name}")
    print(f"Model Artifacts: {model_artifacts}")
    print(f"Training Image: {DEFAULT_CPU_IMAGE}")
    
    print("\nCustom Driver Configuration:")
    print(f"Driver Class: {custom_driver.__class__.__name__}")
    print(f"Process Count Per Node: {custom_driver.process_count_per_node}")
    print(f"Driver Directory: {custom_driver.driver_dir}")
    print(f"Driver Script: {custom_driver.driver_script}")
    
    print("\nHyperparameters Used:")
    for key, value in hyperparameters.items():
        print(f"  {key}: {value}")
    
    print("\n✓ Custom distributed training completed successfully!")
    
else:
    print("Training was not successful.")

Custom Distributed Training Results:
Job Name: custom-distributed-driver-20260716071158
Model Artifacts: s3_model_artifacts='s3://sagemaker-us-west-2-674622101542/custom-distributed-driver/custom-distributed-driver-20260716071158/output/model.tar.gz'
Training Image: 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.0.0-cpu-py310

Custom Driver Configuration:
Driver Class: CustomDriver
Process Count Per Node: 2
Driver Directory: /home/model-server/tmp/tmpoab_oo4k/custom_drivers
Driver Script: driver.py

Hyperparameters Used:
  epochs: 10

✓ Custom distributed training completed successfully!


## Step 8: Clean Up

Clean up temporary files.

In [10]:
try:
    shutil.rmtree(temp_dir)
    print(f"Cleaned up temporary directory: {temp_dir}")
except Exception as e:
    print(f"Could not clean up temp directory: {e}")

print("Cleanup completed!")

Cleaned up temporary directory: /home/model-server/tmp/tmpoab_oo4k
Cleanup completed!


## Summary

This notebook demonstrated:
1. **Custom distributed driver creation**: Extending DistributedConfig for specialized needs
2. **Driver coordination**: How custom drivers manage training processes
3. **ModelTrainer integration**: Seamless integration with SageMaker V3 training
4. **Custom training logic**: Implementing specialized training patterns

Custom distributed drivers provide flexibility for implementing specialized coordination logic, framework integration, and advanced debugging capabilities for distributed training scenarios.